# Notebook 18 — Reviewer-ready multi-seed cross-corpus validation and finalization

This is the **single RunPod notebook** for closing the main statistical and consistency gaps in the
manuscript. It is intentionally separate from the exploratory enhanced FGM+AWP head.

## Scientific contract

- Primary architecture: `csebuetnlp/banglabert` with its standard sequence-classification head.
- Matched systems: vanilla full fine-tuning and full fine-tuning + FGM (`epsilon=0.5`).
- Seeds: `42, 1, 7, 123, 2024`.
- Corpora: `ben_sarc_binary`, `banglasarc_binary`, `banglasarc3_binary`.
- Each source model is selected using **only its source validation split**, then evaluated on all three
  test corpora.
- Off-diagonal evaluation removes normalized exact overlaps with **every source-corpus split** and logs
  both training-seen and conservative all-source removals.
- Uncertainty is calculated across matched training seeds. FGM-vs-vanilla comparisons use an exact
  sign-flip permutation test and Holm correction.
- `09b_fgm_awp_multiseed.csv` is never accepted as multi-seed evidence: it contains one exploratory run.
- Notebook 17's FGM/FGM+AWP architecture conflation is explicitly quarantined.

## Outputs

- Checkpoints: `03_checkpoints/18_multiseed_cross_corpus/`
- Per-run predictions: `04_outputs/predictions/18_multiseed_cross_corpus/`
- Resumable raw tables: `04_outputs/tables/18_*.csv|json`
- Final paper package: `04_outputs/finalized_outputs/{tables,figures}/`

The notebook audits every CSV/JSON under `04_outputs`, but only scientifically compatible artifacts are
used in figures. Superseded or mislabeled artifacts remain untouched and are documented in the audit.

## RunPod quick start

1. Use a PyTorch image and an RTX 4090, RTX 5090, A40, RTX A6000, or A100.
2. Keep the repository layout unchanged; splits must be in `01_data/interim/splits/`.
3. Set `RUNPOD_GPU_HOURLY_USD` if you want the budget guard to use your displayed pod rate.
4. Run all cells. Completed `(system, source, seed)` jobs are skipped safely on rerun.

Expected RTX 4090 wall time is approximately **2–4 hours** using the uploaded project timings; slower
24 GB GPUs may need **4–8 hours**. Runtime is workload- and availability-dependent.


In [1]:
# Install a tested Transformers 4.x stack. Restart the kernel only if pip explicitly requests it.
%pip install -q "transformers>=4.44,<5" "accelerate>=0.33" "safetensors>=0.4" \
    "sentencepiece>=0.2" "scikit-learn>=1.3" "scipy>=1.10" "pandas>=2.0" \
    "matplotlib>=3.7" "nbformat>=5.9"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, gc, json, math, time, random, shutil, hashlib, platform, warnings, unicodedata
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import scipy
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
import transformers

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def find_repo_root():
    env = os.getenv("NB18_ROOT", "").strip()
    candidates = ([Path(env)] if env else []) + [
        Path("/workspace/Sarcasm_detection"), Path.cwd(), *Path.cwd().resolve().parents
    ]
    for c in candidates:
        if (c / "01_data" / "interim" / "splits").exists():
            return c.resolve()
    raise RuntimeError("Repository root not found: need 01_data/interim/splits/.")

ROOT = find_repo_root()
SPLITS = ROOT / "01_data" / "interim" / "splits"
CKPT = ROOT / "03_checkpoints" / "18_multiseed_cross_corpus"
OUT = ROOT / "04_outputs"
TABLES = OUT / "tables"
PRED = OUT / "predictions" / "18_multiseed_cross_corpus"
RUN_LOGS = OUT / "run_logs" / "18_multiseed_cross_corpus"
FINAL = OUT / "finalized_outputs"
FT, FF = FINAL / "tables", FINAL / "figures"
for p in (CKPT, TABLES, PRED, RUN_LOGS, FT, FF):
    p.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "csebuetnlp/banglabert"
CORPORA = ["ben_sarc_binary", "banglasarc_binary", "banglasarc3_binary"]
DISPLAY = {
    "ben_sarc_binary": "Ben-Sarc",
    "banglasarc_binary": "BanglaSarc",
    "banglasarc3_binary": "BanglaSarc3",
}
SEEDS = [42, 1, 7, 123, 2024]
SYSTEMS = {
    "vanilla": {"adversarial": None, "fgm_epsilon": None},
    "fgm": {"adversarial": "fgm", "fgm_epsilon": 0.5},
}

# Matched fixed budget; early stopping observes source validation only.
MAX_LENGTH = 128
EPOCHS = 8
PATIENCE = 2
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
BOOTSTRAP_B = 2000

RUN_TRAINING = True
OVERWRITE_COMPLETED = False
SAVE_CANONICAL_BEST_CHECKPOINT = True   # one validation-selected model per system/source
KEEP_TRAINER_CHECKPOINTS_AFTER_SUCCESS = False

# Cost guard. Override with environment values shown by your RunPod pod page.
HOURLY_RATE_USD = float(os.getenv("RUNPOD_GPU_HOURLY_USD", "0.69"))
MAX_COMPUTE_USD = float(os.getenv("NB18_MAX_COMPUTE_USD", "9.00"))
EXPECTED_FIRST_JOB_SECONDS = float(os.getenv("NB18_FIRST_JOB_SECONDS", "600"))

RUN_TABLE = TABLES / "18_transformer_cross_corpus_multiseed_runs.csv"
SUMMARY_TABLE = TABLES / "18_transformer_cross_corpus_multiseed_summary.csv"
TEST_TABLE = TABLES / "18_fgm_vs_vanilla_paired_seed_tests.csv"
GAP_TABLE = TABLES / "18_transfer_gaps_by_seed.csv"
SHIFT_TABLE = TABLES / "18_cross_corpus_shift_audit.csv"

HAS_CUDA = torch.cuda.is_available()
HAS_BF16 = HAS_CUDA and torch.cuda.is_bf16_supported()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else "CPU"
if RUN_TRAINING and not HAS_CUDA:
    raise RuntimeError("RUN_TRAINING=True requires a CUDA GPU. Use a RunPod GPU pod.")

print("ROOT       :", ROOT)
print("SPLITS     :", SPLITS)
print("GPU        :", GPU_NAME, "| bf16:", HAS_BF16)
print("jobs       :", len(SEEDS) * len(SYSTEMS) * len(CORPORA), "training runs")
print("budget cap : $", MAX_COMPUTE_USD, "at", HOURLY_RATE_USD, "/hour")


ROOT       : /workspace/Sarcasm_detection
SPLITS     : /workspace/Sarcasm_detection/01_data/interim/splits
GPU        : NVIDIA GeForce RTX 4090 | bf16: True
jobs       : 30 training runs
budget cap : $ 9.0 at 0.69 /hour


In [3]:
# Reproducibility, data contracts, and leakage controls
_ZW = dict.fromkeys(map(ord, ["\u200b", "\u200c", "\u200d", "\ufeff"]), None)

def norm_key(value):
    if not isinstance(value, str):
        value = "" if pd.isna(value) else str(value)
    value = unicodedata.normalize("NFC", value).translate(_ZW)
    return re.sub(r"\s+", " ", value).strip().casefold()

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if HAS_CUDA:
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

def item_id(text, gold):
    blob = f"{norm_key(text)}\n{int(gold)}".encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:20]

def read_split(corpus, split):
    path = SPLITS / f"{corpus}_{split}.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    need = {"text", "label_binary"}
    if not need.issubset(df.columns):
        raise ValueError(f"{path.name}: expected columns {sorted(need)}, got {list(df.columns)}")
    df = df.dropna(subset=["text", "label_binary"]).copy()
    df["text"] = df["text"].astype(str)
    df["label_binary"] = df["label_binary"].astype(int)
    if not set(df["label_binary"].unique()).issubset({0, 1}):
        raise ValueError(f"{path.name}: non-binary labels detected")
    df["norm_key_nb18"] = df["text"].map(norm_key)
    df["item_id"] = [item_id(t, y) for t, y in zip(df.text, df.label_binary)]
    if df["norm_key_nb18"].duplicated().any():
        raise ValueError(f"{path.name}: normalized duplicates remain inside split")
    return df.reset_index(drop=True)

DATA = {c: {s: read_split(c, s) for s in ("train", "val", "test")} for c in CORPORA}

def assert_internal_disjoint(corpus):
    keys = {s: set(DATA[corpus][s]["norm_key_nb18"]) for s in ("train", "val", "test")}
    assert not (keys["train"] & keys["val"])
    assert not (keys["train"] & keys["test"])
    assert not (keys["val"] & keys["test"])

for corpus in CORPORA:
    assert_internal_disjoint(corpus)

def clean_target(source, target):
    # Conservative cross-corpus cleaning; diagonal test data are unchanged.
    te = DATA[target]["test"].copy()
    if source == target:
        return te, dict(n_eval_total=len(te), n_removed_train_seen=0,
                        n_removed_all_source=0, n_eval_clean=len(te))
    train_seen = set(DATA[source]["train"]["norm_key_nb18"]) | set(DATA[source]["val"]["norm_key_nb18"])
    all_source = train_seen | set(DATA[source]["test"]["norm_key_nb18"])
    k = te["norm_key_nb18"]
    removed_train = int(k.isin(train_seen).sum())
    keep = ~k.isin(all_source)
    clean = te.loc[keep].reset_index(drop=True)
    audit = dict(n_eval_total=len(te), n_removed_train_seen=removed_train,
                 n_removed_all_source=int((~keep).sum()), n_eval_clean=len(clean))
    if set(clean["norm_key_nb18"]) & all_source:
        raise AssertionError(f"cross-corpus leakage remains: {source} -> {target}")
    return clean, audit

split_rows = []
for c in CORPORA:
    for s in ("train", "val", "test"):
        d = DATA[c][s]
        split_rows.append(dict(corpus=c, split=s, n=len(d), positive_rate=d.label_binary.mean()))
split_audit = pd.DataFrame(split_rows)
split_audit.to_csv(TABLES / "18_split_audit.csv", index=False)
display(split_audit)


,corpus,split,n,positive_rate
0,ben_sarc_binary,train,20498,0.500000
1,ben_sarc_binary,val,2562,0.500000
2,ben_sarc_binary,test,2563,0.500195
3,banglasarc_binary,train,3708,0.358145
4,banglasarc_binary,val,463,0.358531
5,banglasarc_binary,test,464,0.357759
6,banglasarc3_binary,train,6328,0.499210
7,banglasarc3_binary,val,791,0.499368
8,banglasarc3_binary,test,791,0.499368


In [ ]:
# Model, FGM trainer, metrics, and serialization
class EncodedDataset(torch.utils.data.Dataset):
    def __init__(self, frame, tokenizer, max_length=128):
        self.frame = frame.reset_index(drop=True)
        self.enc = tokenizer(
            self.frame["text"].tolist(), truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt"
        )
        self.labels = torch.tensor(self.frame["label_binary"].values, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        out = {k: v[idx] for k, v in self.enc.items()}
        out["labels"] = self.labels[idx]
        return out

class FGM:
    def __init__(self, model, epsilon=0.5, embedding_name="word_embeddings"):
        self.model, self.epsilon, self.embedding_name = model, epsilon, embedding_name
        self.backup = {}
    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.embedding_name in name and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if torch.isfinite(norm) and norm.item() > 0:
                    param.data.add_(self.epsilon * param.grad / norm)
    def restore(self):
        for name, value in self.backup.items():
            dict(self.model.named_parameters())[name].data.copy_(value)
        self.backup.clear()

class FGMTrainer(Trainer):
    def __init__(self, *args, fgm_epsilon=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.fgm = FGM(self.model, epsilon=fgm_epsilon)
    def training_step(self, model, inputs, num_items_in_batch=None):
        model.train()
        inputs = self._prepare_inputs(inputs)
        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1:
            loss = loss.mean()
        self.accelerator.backward(loss)
        self.fgm.attack()
        with self.compute_loss_context_manager():
            adversarial_loss = self.compute_loss(model, inputs)
        if self.args.n_gpu > 1:
            adversarial_loss = adversarial_loss.mean()
        self.accelerator.backward(adversarial_loss)
        self.fgm.restore()
        return loss.detach()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.asarray(logits).argmax(axis=-1)
    return {"macro_f1": f1_score(labels, pred, average="macro", zero_division=0),
            "accuracy": accuracy_score(labels, pred)}

def softmax_np(logits):
    z = np.asarray(logits, dtype=np.float64)
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def score(gold, pred):
    class_recall = recall_score(gold, pred, labels=[0, 1], average=None, zero_division=0)
    return dict(
        accuracy=float(accuracy_score(gold, pred)),
        macro_f1=float(f1_score(gold, pred, average="macro", zero_division=0)),
        weighted_f1=float(f1_score(gold, pred, average="weighted", zero_division=0)),
        precision_macro=float(precision_score(gold, pred, average="macro", zero_division=0)),
        recall_macro=float(recall_score(gold, pred, average="macro", zero_division=0)),
        recall_class_0=float(class_recall[0]), recall_class_1=float(class_recall[1]),
    )

def expected_calibration_error(gold, probs, n_bins=15):
    gold = np.asarray(gold); probs = np.asarray(probs)
    conf = probs.max(axis=1); pred = probs.argmax(axis=1)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs((pred[mask] == gold[mask]).mean() - conf[mask].mean())
    return float(ece)

def multiclass_brier(gold, probs):
    onehot = np.eye(probs.shape[1], dtype=float)[np.asarray(gold, dtype=int)]
    return float(np.mean(np.sum((np.asarray(probs) - onehot) ** 2, axis=1)))

def fit_temperature(logits, gold):
    logits = np.asarray(logits, dtype=np.float64); gold = np.asarray(gold, dtype=int)
    def objective(log_temperature):
        p = softmax_np(logits / np.exp(log_temperature))
        return float(-np.log(np.clip(p[np.arange(len(gold)), gold], 1e-12, 1.0)).mean())
    result = minimize_scalar(objective, bounds=(-2.3, 2.3), method="bounded")
    return float(np.exp(result.x))

def make_args(output_dir, seed, adversarial=False):
    # BF16 is preferred for FGM stability on Ampere/Ada. FGM falls back to FP32 on older GPUs.
    use_bf16 = bool(HAS_BF16)
    use_fp16 = bool(HAS_CUDA and not use_bf16 and not adversarial)
    common = dict(
        output_dir=str(output_dir), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="linear", save_strategy="epoch", logging_strategy="epoch",
        save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, report_to="none", seed=seed, data_seed=seed,
        bf16=use_bf16, fp16=use_fp16, dataloader_num_workers=2,
        remove_unused_columns=True,
    )
    try:
        return TrainingArguments(eval_strategy="epoch", **common)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **common)

def save_prediction_file(path, frame, logits, system, source, target, seed, temperature):
    probs = softmax_np(logits)
    calibrated = softmax_np(np.asarray(logits) / temperature)
    pred = probs.argmax(axis=1)
    out = pd.DataFrame({
        "item_id": frame["item_id"].values,
        "gold_label": frame["label_binary"].values,
        "pred_label": pred,
        "correct": (pred == frame["label_binary"].values).astype(int),
        "logit_0": np.asarray(logits)[:, 0], "logit_1": np.asarray(logits)[:, 1],
        "prob_0": probs[:, 0], "prob_1": probs[:, 1],
        "prob_cal_0": calibrated[:, 0], "prob_cal_1": calibrated[:, 1],
        "temperature": temperature,
        "system": system, "source": source, "target": target, "seed": seed,
    })
    path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(path, index=False)
    return out

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print("tokenizer:", tokenizer.__class__.__name__)


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer: ElectraTokenizerFast


In [5]:
# Resume state and budget estimation
def load_run_rows():
    if not RUN_TABLE.exists():
        return []
    old = pd.read_csv(RUN_TABLE)
    return old.to_dict("records")

run_rows = load_run_rows()

def completed_jobs(rows):
    if not rows:
        return set()
    d = pd.DataFrame(rows)
    done = set()
    for key, g in d.groupby(["system", "source", "seed"]):
        if set(g["target"]) == set(CORPORA):
            system, source, seed = key
            expected = [PRED / f"18_{system}_{source}_seed{int(seed)}_to_{t}.csv" for t in CORPORA]
            if all(p.exists() for p in expected):
                done.add((str(system), str(source), int(seed)))
    return done

DONE = completed_jobs(run_rows)
print("completed training jobs:", len(DONE), "/", len(SYSTEMS) * len(CORPORA) * len(SEEDS))

def runtime_samples(rows):
    if not rows:
        return []
    d = pd.DataFrame(rows)
    return d.drop_duplicates(["system", "source", "seed"])["train_seconds"].dropna().tolist()

NOTEBOOK_START = time.time()

def enforce_budget(rows, pending_jobs):
    samples = runtime_samples(rows)
    per_job = float(np.median(samples)) if samples else EXPECTED_FIRST_JOB_SECONDS
    elapsed_gpu_h = sum(samples) / 3600.0
    projected_h = elapsed_gpu_h + pending_jobs * per_job / 3600.0
    projected_cost = projected_h * HOURLY_RATE_USD
    print(f"projected compute: {projected_h:.2f} GPU-h, ${projected_cost:.2f}")
    if projected_cost > MAX_COMPUTE_USD:
        raise RuntimeError(
            f"Budget guard stopped before the next job: projected ${projected_cost:.2f} "
            f"> cap ${MAX_COMPUTE_USD:.2f}. Raise NB18_MAX_COMPUTE_USD only after checking the pod rate."
        )


completed training jobs: 0 / 30


In [6]:
# Main matched multi-seed experiment (30 resumable training jobs)
def maybe_save_canonical(trainer, system, source, seed, val_macro_f1):
    if not SAVE_CANONICAL_BEST_CHECKPOINT:
        return None
    dest = CKPT / system / source / "best_by_validation"
    record = dest / "selection.json"
    previous = None
    if record.exists():
        try:
            previous = json.load(open(record))
        except Exception:
            previous = None
    if previous is not None and float(previous.get("val_macro_f1", -1)) >= val_macro_f1:
        return str(dest)
    dest.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(dest))
    tokenizer.save_pretrained(str(dest))
    payload = dict(system=system, source=source, seed=int(seed),
                   val_macro_f1=float(val_macro_f1), selected_without_test=True,
                   model_name=MODEL_NAME, saved_at=datetime.now(timezone.utc).isoformat())
    json.dump(payload, open(record, "w"), indent=2)
    return str(dest)

def train_and_evaluate(system, source, seed):
    cfg = SYSTEMS[system]
    set_seed(seed)
    run_id = f"18_{system}_{source}_seed{seed}"
    trainer_dir = CKPT / "_trainer_state" / run_id
    trainer_dir.mkdir(parents=True, exist_ok=True)

    tr = DATA[source]["train"]
    va = DATA[source]["val"]
    tr_ds = EncodedDataset(tr, tokenizer, MAX_LENGTH)
    va_ds = EncodedDataset(va, tokenizer, MAX_LENGTH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = make_args(trainer_dir, seed, adversarial=(system == "fgm"))
    trainer_cls = FGMTrainer if system == "fgm" else Trainer
    kwargs = dict(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
                  compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    if system == "fgm":
        kwargs["fgm_epsilon"] = float(cfg["fgm_epsilon"])
    trainer = trainer_cls(**kwargs)

    checkpoints = sorted(trainer_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    resume = str(checkpoints[-1]) if checkpoints else None
    print(f"\n[{run_id}] train n={len(tr):,}, val n={len(va):,}, resume={bool(resume)}")
    t0 = time.time()
    trainer.train(resume_from_checkpoint=resume)
    train_seconds = float(time.time() - t0)
    val_output = trainer.predict(va_ds)
    val_pred = np.asarray(val_output.predictions).argmax(axis=1)
    val_metrics = score(va.label_binary.values, val_pred)
    temperature = fit_temperature(val_output.predictions, va.label_binary.values)
    best_checkpoint = maybe_save_canonical(trainer, system, source, seed, val_metrics["macro_f1"])

    rows = []
    for target in CORPORA:
        clean, audit = clean_target(source, target)
        ds = EncodedDataset(clean, tokenizer, MAX_LENGTH)
        output = trainer.predict(ds)
        logits = np.asarray(output.predictions)
        pred = logits.argmax(axis=1)
        metrics = score(clean.label_binary.values, pred)
        probs = softmax_np(logits)
        calibrated = softmax_np(logits / temperature)
        cm = confusion_matrix(clean.label_binary.values, pred, labels=[0, 1]).tolist()
        pred_path = PRED / f"18_{system}_{source}_seed{seed}_to_{target}.csv"
        save_prediction_file(pred_path, clean, logits, system, source, target, seed, temperature)
        rows.append(dict(
            run_id=run_id, system=system, architecture="BanglaBERT standard classification head",
            adversarial=(system == "fgm"), fgm_epsilon=cfg["fgm_epsilon"],
            model_name=MODEL_NAME, source=source, target=target, seed=int(seed),
            in_domain=bool(source == target), n_train=len(tr), n_val=len(va),
            **audit, val_macro_f1=val_metrics["macro_f1"], val_accuracy=val_metrics["accuracy"],
            test_macro_f1=metrics["macro_f1"], test_accuracy=metrics["accuracy"],
            test_weighted_f1=metrics["weighted_f1"],
            test_precision_macro=metrics["precision_macro"], test_recall_macro=metrics["recall_macro"],
            test_recall_class_0=metrics["recall_class_0"], test_recall_class_1=metrics["recall_class_1"],
            temperature=temperature,
            ece=expected_calibration_error(clean.label_binary.values, probs),
            ece_temperature_scaled=expected_calibration_error(clean.label_binary.values, calibrated),
            brier=multiclass_brier(clean.label_binary.values, probs),
            brier_temperature_scaled=multiclass_brier(clean.label_binary.values, calibrated),
            confusion_matrix=json.dumps(cm), train_seconds=train_seconds,
            best_checkpoint=best_checkpoint, prediction_path=str(pred_path.relative_to(ROOT)),
        ))
        print(f"  {DISPLAY[source]} -> {DISPLAY[target]}: F1={metrics['macro_f1']:.4f} "
              f"(clean n={len(clean)}, removed={audit['n_removed_all_source']})")

    del trainer, model, tr_ds, va_ds
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
    if not KEEP_TRAINER_CHECKPOINTS_AFTER_SUCCESS:
        shutil.rmtree(trainer_dir, ignore_errors=True)
    return rows

if RUN_TRAINING:
    jobs = [(s, c, seed) for s in SYSTEMS for c in CORPORA for seed in SEEDS]
    for job_index, (system, source, seed) in enumerate(jobs, start=1):
        key = (system, source, int(seed))
        if key in DONE and not OVERWRITE_COMPLETED:
            print("skip completed:", key)
            continue
        pending = sum((ss, cc, int(sd)) not in DONE for ss, cc, sd in jobs[job_index-1:])
        enforce_budget(run_rows, pending)
        new_rows = train_and_evaluate(system, source, seed)
        # Replace a partially recorded job atomically at table level.
        run_rows = [r for r in run_rows if (str(r.get("system")), str(r.get("source")), int(r.get("seed", -1))) != key]
        run_rows.extend(new_rows)
        pd.DataFrame(run_rows).to_csv(RUN_TABLE, index=False)
        DONE.add(key)
        print(f"saved {RUN_TABLE.name}: {len(run_rows)} source-target rows")

print("training stage complete; rows:", len(run_rows))


projected compute: 5.00 GPU-h, $3.45


pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[18_vanilla_ben_sarc_binary_seed42] train n=20,498, val n=2,562, resume=False


model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

...


[18_fgm_banglasarc3_binary_seed2024] train n=6,328, val n=791, resume=False


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.625700,0.575537,0.696082,0.701643
2,0.481800,0.509894,0.753635,0.754741
3,0.331400,0.553186,0.755780,0.756005
4,0.187400,0.632021,0.762308,0.762326
5,0.104800,0.735076,0.759428,0.759798
6,0.059300,0.853522,0.752209,0.752212


  BanglaSarc3 -> Ben-Sarc: F1=0.6673 (clean n=2563, removed=0)


  BanglaSarc3 -> BanglaSarc: F1=0.6458 (clean n=436, removed=28)


  BanglaSarc3 -> BanglaSarc3: F1=0.7686 (clean n=791, removed=0)
saved 18_transformer_cross_corpus_multiseed_runs.csv: 90 source-target rows
training stage complete; rows: 90


In [7]:
# Validate completeness and aggregate uncertainty across seeds
runs = pd.read_csv(RUN_TABLE)
expected_rows = len(SYSTEMS) * len(CORPORA) * len(CORPORA) * len(SEEDS)
assert len(runs) == expected_rows, f"Expected {expected_rows} rows, found {len(runs)}"
assert set(runs.system) == set(SYSTEMS)
assert set(runs.source) == set(CORPORA) == set(runs.target)
assert set(runs.seed.astype(int)) == set(SEEDS)
assert runs.groupby(["system", "source", "seed"]).target.nunique().eq(len(CORPORA)).all()
assert np.isfinite(runs[["test_macro_f1", "test_accuracy"]].values).all()

def mean_ci(values, confidence=0.95):
    x = np.asarray(values, dtype=float)
    n = len(x); mean = float(x.mean())
    sd = float(x.std(ddof=1)) if n > 1 else 0.0
    if n < 2 or sd == 0:
        return mean, sd, mean, mean
    half = float(stats.t.ppf((1 + confidence) / 2, n - 1) * sd / math.sqrt(n))
    return mean, sd, mean - half, mean + half

summary_rows = []
for key, g in runs.groupby(["system", "source", "target", "in_domain"], sort=True):
    system, source, target, in_domain = key
    mf, sf, lf, hf = mean_ci(g.test_macro_f1)
    ma, sa, la, ha = mean_ci(g.test_accuracy)
    summary_rows.append(dict(
        system=system, source=source, target=target, in_domain=bool(in_domain), seeds=len(g),
        macro_f1_mean=mf, macro_f1_std=sf, macro_f1_ci95_lo=lf, macro_f1_ci95_hi=hf,
        accuracy_mean=ma, accuracy_std=sa, accuracy_ci95_lo=la, accuracy_ci95_hi=ha,
        ece_mean=float(g.ece.mean()), ece_std=float(g.ece.std(ddof=1)),
        ece_temperature_scaled_mean=float(g.ece_temperature_scaled.mean()),
        brier_mean=float(g.brier.mean()),
        brier_temperature_scaled_mean=float(g.brier_temperature_scaled.mean()),
        recall_class_0_mean=float(g.test_recall_class_0.mean()),
        recall_class_1_mean=float(g.test_recall_class_1.mean()),
        n_eval_total=int(g.n_eval_total.iloc[0]),
        n_removed_train_seen=int(g.n_removed_train_seen.iloc[0]),
        n_removed_all_source=int(g.n_removed_all_source.iloc[0]),
        n_eval_clean=int(g.n_eval_clean.iloc[0]),
    ))
summary = pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_TABLE, index=False)

# Paired per-seed transfer gaps: source diagonal minus the same source's off-diagonal target.
gap_rows = []
for (system, source, seed), g in runs.groupby(["system", "source", "seed"]):
    diag = float(g.loc[g.target == source, "test_macro_f1"].iloc[0])
    for r in g.itertuples():
        if r.target != source:
            gap_rows.append(dict(system=system, source=source, target=r.target, seed=int(seed),
                                 in_domain_macro_f1=diag, transfer_macro_f1=r.test_macro_f1,
                                 absolute_drop=diag-r.test_macro_f1,
                                 relative_drop=(diag-r.test_macro_f1)/diag if diag else np.nan))
gaps = pd.DataFrame(gap_rows)
gaps.to_csv(GAP_TABLE, index=False)

display(summary.round(4))


,system,source,target,in_domain,seeds,macro_f1_mean,macro_f1_std,macro_f1_ci95_lo,macro_f1_ci95_hi,accuracy_mean,...,ece_std,ece_temperature_scaled_mean,brier_mean,brier_temperature_scaled_mean,recall_class_0_mean,recall_class_1_mean,n_eval_total,n_removed_train_seen,n_removed_all_source,n_eval_clean
0,fgm,banglasarc3_binary,banglasarc3_binary,True,5,0.7595,0.0089,0.7485,0.7705,0.7598,...,0.0368,0.0380,0.3378,0.3251,0.7419,0.7777,791,0,0,791
1,fgm,banglasarc3_binary,banglasarc_binary,False,5,0.6238,0.0464,0.5662,0.6814,0.6339,...,0.0328,0.1145,0.5138,0.4804,0.6400,0.6241,464,24,28,436
2,fgm,banglasarc3_binary,ben_sarc_binary,False,5,0.6700,0.0052,0.6636,0.6764,0.6719,...,0.0609,0.0657,0.4582,0.4295,0.6048,0.7390,2563,0,0,2563
3,fgm,banglasarc_binary,banglasarc3_binary,False,5,0.3341,0.0042,0.3288,0.3393,0.4850,...,0.0036,0.5069,1.0197,1.0129,0.9871,0.0087,791,25,29,762
4,fgm,banglasarc_binary,banglasarc_binary,True,5,0.9817,0.0026,0.9785,0.9848,0.9832,...,0.0023,0.0107,0.0292,0.0283,0.9906,0.9699,464,0,0,464
5,fgm,banglasarc_binary,ben_sarc_binary,False,5,0.3460,0.0044,0.3405,0.3515,0.4994,...,0.0031,0.4922,0.9920,0.9857,0.9841,0.0151,2563,0,0,2563
6,fgm,ben_sarc_binary,banglasarc3_binary,False,5,0.6490,0.0086,0.6384,0.6597,0.6493,...,0.0526,0.1412,0.4951,0.4741,0.6652,0.6334,791,0,0,791
7,fgm,ben_sarc_binary,banglasarc_binary,False,5,0.5956,0.0351,0.5519,0.6392,0.6353,...,0.0413,0.1618,0.5048,0.4851,0.7362,0.4542,464,0,0,464
8,fgm,ben_sarc_binary,ben_sarc_binary,True,5,0.7984,0.0035,0.7941,0.8027,0.7985,...,0.0447,0.0215,0.2905,0.2830,0.8100,0.7871,2563,0,0,2563
9,vanilla,banglasarc3_binary,banglasarc3_binary,True,5,0.7605,0.0084,0.7501,0.7710,0.7611,...,0.0409,0.0458,0.3635,0.3336,0.7571,0.7651,791,0,0,791


In [8]:
# Exact matched-seed FGM-vs-vanilla tests with Holm correction
def exact_sign_flip_test(differences):
    d = np.asarray(differences, dtype=float)
    observed = abs(d.mean())
    vals = []
    for bits in range(2 ** len(d)):
        signs = np.array([1 if (bits >> i) & 1 else -1 for i in range(len(d))])
        vals.append(abs((d * signs).mean()))
    return float(np.mean(np.asarray(vals) >= observed - 1e-15))

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p); m = len(p); adjusted = np.empty(m); running = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (m-rank) * p[idx])
        running = max(running, value)
        adjusted[idx] = running
    return adjusted

test_rows = []
for (source, target), g in runs.groupby(["source", "target"]):
    piv = g.pivot(index="seed", columns="system", values="test_macro_f1").loc[SEEDS]
    diff = piv["fgm"] - piv["vanilla"]
    md, sd, lo, hi = mean_ci(diff.values)
    test_rows.append(dict(source=source, target=target, in_domain=source==target,
                          n_seeds=len(diff), vanilla_mean=piv.vanilla.mean(), fgm_mean=piv.fgm.mean(),
                          delta_mean=md, delta_std=sd, delta_ci95_lo=lo, delta_ci95_hi=hi,
                          exact_sign_flip_p=exact_sign_flip_test(diff.values)))
paired = pd.DataFrame(test_rows)
paired["p_holm_9_cells"] = holm_adjust(paired.exact_sign_flip_p.values)
paired["significant_holm_0_05"] = paired.p_holm_9_cells < 0.05
paired.to_csv(TEST_TABLE, index=False)
display(paired.round(5))


,source,target,in_domain,n_seeds,vanilla_mean,fgm_mean,delta_mean,delta_std,delta_ci95_lo,delta_ci95_hi,exact_sign_flip_p,p_holm_9_cells,significant_holm_0_05
0,banglasarc3_binary,banglasarc3_binary,True,5,0.76053,0.75952,-0.00100,0.01386,-0.01821,0.01620,0.9375,1.0,False
1,banglasarc3_binary,banglasarc_binary,False,5,0.64147,0.62380,-0.01767,0.02190,-0.04486,0.00953,0.1875,1.0,False
2,banglasarc3_binary,ben_sarc_binary,False,5,0.66753,0.66997,0.00244,0.00425,-0.00283,0.00772,0.3125,1.0,False
3,banglasarc_binary,banglasarc3_binary,False,5,0.33736,0.33405,-0.00331,0.00466,-0.00909,0.00247,0.2500,1.0,False
4,banglasarc_binary,banglasarc_binary,True,5,0.97280,0.98165,0.00885,0.00581,0.00164,0.01607,0.1250,1.0,False
5,banglasarc_binary,ben_sarc_binary,False,5,0.34618,0.34602,-0.00016,0.00291,-0.00378,0.00346,0.9375,1.0,False
6,ben_sarc_binary,banglasarc3_binary,False,5,0.64411,0.64902,0.00491,0.00926,-0.00659,0.01642,0.5000,1.0,False
7,ben_sarc_binary,banglasarc_binary,False,5,0.59064,0.59558,0.00494,0.02333,-0.02403,0.03391,0.6875,1.0,False
8,ben_sarc_binary,ben_sarc_binary,True,5,0.79141,0.79840,0.00699,0.00626,-0.00079,0.01476,0.1250,1.0,False


In [9]:
# Dataset-shift audit (CPU-only; descriptive, not used for model selection)
def token_set(texts):
    out = set()
    for t in texts:
        out.update(norm_key(t).split())
    return out

shift_rows = []
for source in CORPORA:
    src = DATA[source]["train"]
    src_tokens = token_set(src.text)
    src_all_keys = set().union(*(set(DATA[source][s].norm_key_nb18) for s in ("train", "val", "test")))
    for target in CORPORA:
        tgt, audit = clean_target(source, target)
        tgt_tokens = token_set(tgt.text)
        union = src_tokens | tgt_tokens
        token_jaccard = len(src_tokens & tgt_tokens) / len(union) if union else 0.0
        target_oov = len(tgt_tokens - src_tokens) / len(tgt_tokens) if tgt_tokens else 0.0

        # Shared descriptive representation fitted on both domains; never enters training.
        vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                              max_features=25000, sublinear_tf=True, dtype=np.float32)
        combined = pd.concat([src.text, tgt.text], ignore_index=True)
        X = vec.fit_transform(combined)
        a = np.asarray(X[:len(src)].mean(axis=0))
        b = np.asarray(X[len(src):].mean(axis=0))
        char_cosine = float(cosine_similarity(a, b)[0, 0])
        shift_rows.append(dict(
            source=source, target=target, in_domain=source==target,
            token_jaccard=token_jaccard, target_token_oov_rate=target_oov,
            char_tfidf_centroid_cosine=char_cosine,
            source_train_positive_rate=float(src.label_binary.mean()),
            target_test_positive_rate=float(tgt.label_binary.mean()),
            positive_rate_abs_gap=abs(float(src.label_binary.mean()-tgt.label_binary.mean())),
            cross_corpus_exact_overlap_all_source=int(audit["n_removed_all_source"]),
            n_target_clean=int(audit["n_eval_clean"]),
        ))
shift = pd.DataFrame(shift_rows)
shift.to_csv(SHIFT_TABLE, index=False)

off = summary[(summary.system == "fgm") & (~summary.in_domain)].merge(
    shift, on=["source", "target", "in_domain"], how="left"
)
rho, p = stats.spearmanr(off.char_tfidf_centroid_cosine, off.macro_f1_mean)
shift_note = dict(
    analysis="exploratory Spearman association over six directed off-diagonal cells",
    n_cells=int(len(off)), rho=float(rho), p_value=float(p),
    caution="descriptive association; too few domain pairs for causal inference",
)
json.dump(shift_note, open(TABLES / "18_shift_transfer_association.json", "w"), indent=2)
display(shift.round(4))
print(shift_note)


,source,target,in_domain,token_jaccard,target_token_oov_rate,char_tfidf_centroid_cosine,source_train_positive_rate,target_test_positive_rate,positive_rate_abs_gap,cross_corpus_exact_overlap_all_source,n_target_clean
0,ben_sarc_binary,ben_sarc_binary,True,0.1982,0.2453,0.9869,0.5000,0.5002,0.0002,0,2563
1,ben_sarc_binary,banglasarc_binary,False,0.0543,0.3627,0.7801,0.5000,0.3578,0.1422,0,464
2,ben_sarc_binary,banglasarc3_binary,False,0.0939,0.1994,0.9213,0.5000,0.4994,0.0006,0,791
3,banglasarc_binary,ben_sarc_binary,False,0.2023,0.6191,0.8278,0.3581,0.5002,0.1421,0,2563
4,banglasarc_binary,banglasarc_binary,True,0.1300,0.4204,0.9314,0.3581,0.3578,0.0004,0,464
5,banglasarc_binary,banglasarc3_binary,False,0.1651,0.4353,0.8492,0.3581,0.5131,0.1550,29,762
6,banglasarc3_binary,ben_sarc_binary,False,0.2559,0.4725,0.9149,0.4992,0.5002,0.0010,0,2563
7,banglasarc3_binary,banglasarc_binary,False,0.1019,0.4290,0.7917,0.4992,0.3807,0.1185,28,436
8,banglasarc3_binary,banglasarc3_binary,True,0.1820,0.2743,0.9586,0.4992,0.4994,0.0002,0,791


{'analysis': 'exploratory Spearman association over six directed off-diagonal cells', 'n_cells': 6, 'rho': 0.4285714285714286, 'p_value': 0.3965014577259473, 'caution': 'descriptive association; too few domain pairs for causal inference'}


In [10]:
# Full CSV/JSON inventory and explicit source-of-truth decisions
def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def classify_artifact(rel, rows, cols):
    name = Path(rel).name
    low = rel.lower()
    if name == "09b_fgm_awp_multiseed.csv" and (rows is None or rows < 2):
        return "excluded", "single exploratory FGM+AWP run mislabeled as multiseed"
    if name in {"17_proposed_headline.csv", "17_proposed_multiseed.csv"}:
        return "excluded", "Notebook 17 conflates plain-FGM seeds with the distinct FGM+AWP head"
    if name in {"10_cross_dataset_matrix.csv", "10_cross_dataset_pivot.csv"}:
        return "superseded", "single-seed transformer transfer result; replaced by Notebook 18"
    if name.startswith("18_transformer_cross_corpus") or name in {
        "18_fgm_vs_vanilla_paired_seed_tests.csv", "18_transfer_gaps_by_seed.csv",
        "18_cross_corpus_shift_audit.csv", "18_split_audit.csv",
    }:
        return "canonical", "Notebook 18 reviewer-ready evidence"
    canonical_inputs = {
        "01_split_summary.csv", "02_classical_ml_results.csv", "03_dl_results.csv",
        "03_dl_reference_gap_same_split.csv", "04_frozen_transfer_results.csv",
        "06_multi_backbone_summary.csv", "07_adversarial_summary.csv", "09_final_multiseed.csv",
        "09b_fgm_awp_summary.csv", "11_ensemble_results.csv", "13_calibration.csv",
        "14_length_stratified.csv", "18_llm_baselines.csv",
    }
    if name in canonical_inputs and "finalized_outputs" not in low:
        return "canonical_input", "used in a consistency-resolved figure or evidence audit"
    if "finalized_outputs" in low:
        return "legacy_finalized", "retained for provenance; Notebook 18 writes corrected replacements"
    if "prediction" in low:
        return "prediction_evidence", "retained for reproducibility and downstream paired analyses"
    return "inventory_only", "parsed and recorded; not scientifically compatible with a current figure"

inventory = []
for path in sorted(OUT.rglob("*")):
    if not path.is_file() or path.suffix.lower() not in {".csv", ".json"}:
        continue
    if "__MACOSX" in path.parts or path.name.startswith("._"):
        continue
    rel = str(path.relative_to(OUT))
    rows = None; cols = None; parse_error = None
    try:
        if path.suffix.lower() == ".csv":
            d = pd.read_csv(path)
            rows, cols = len(d), list(d.columns)
        else:
            obj = json.load(open(path))
            rows = len(obj) if hasattr(obj, "__len__") else 1
            cols = list(obj.keys()) if isinstance(obj, dict) else None
    except Exception as exc:
        parse_error = repr(exc)
    decision, reason = classify_artifact(rel, rows, cols)
    inventory.append(dict(
        path=rel, kind=path.suffix.lower().lstrip("."), bytes=path.stat().st_size,
        sha256=sha256_file(path), rows=rows,
        columns=json.dumps(cols, ensure_ascii=False) if cols is not None else None,
        parse_error=parse_error, decision=decision, reason=reason,
    ))

inventory = pd.DataFrame(inventory)
inventory["duplicate_content"] = inventory.duplicated("sha256", keep=False)
inventory.to_csv(TABLES / "18_artifact_inventory.csv", index=False)
decisions = inventory[["path", "decision", "reason", "duplicate_content", "parse_error"]].copy()
decisions.to_csv(TABLES / "18_artifact_decisions.csv", index=False)

assert not inventory.parse_error.notna().any(), "At least one CSV/JSON failed to parse; inspect artifact inventory."
assert ((inventory.path.str.endswith("09b_fgm_awp_multiseed.csv")) &
        (inventory.decision == "excluded")).any(), "mislabeled 09b multiseed artifact was not quarantined"
print(inventory.decision.value_counts().to_string())


decision
prediction_evidence    90
canonical               6
canonical_input         4
legacy_finalized        1
excluded                1
inventory_only          1


In [11]:
# Publication figure engine: IEEE-column dimensions, vector PDF + 600-dpi PNG
C = dict(
    blue="#0072B2", orange="#E69F00", green="#009E73", vermillion="#D55E00",
    purple="#CC79A7", sky="#56B4E9", yellow="#F0E442", gray="#777777",
    light="#D9E2E8", dark="#263238",
)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8.2, "axes.titlesize": 9.0,
    "axes.labelsize": 8.2, "axes.linewidth": 0.8, "axes.edgecolor": "#333333",
    "xtick.labelsize": 7.2, "ytick.labelsize": 7.2, "legend.fontsize": 7.0,
    "figure.facecolor": "white", "axes.facecolor": "white",
})
IN1, IN2 = 3.5, 7.16
generated_figures = []

def savefig(fig, stem, aliases=()):
    for name in (stem, *aliases):
        for ext in ("pdf", "png"):
            fig.savefig(FF / f"{name}.{ext}")
        generated_figures.append(name)
    plt.close(fig)
    print("figure:", stem)

def heatmap_mean_std(system, stem, title):
    d = summary[summary.system == system]
    mean = d.pivot(index="source", columns="target", values="macro_f1_mean").loc[CORPORA, CORPORA]
    std = d.pivot(index="source", columns="target", values="macro_f1_std").loc[CORPORA, CORPORA]
    fig, ax = plt.subplots(figsize=(IN1, 3.05))
    im = ax.imshow(mean.values, cmap="viridis", vmin=0.30, vmax=1.00, aspect="auto")
    ax.set_xticks(range(3)); ax.set_xticklabels([DISPLAY[c] for c in CORPORA], rotation=25, ha="right")
    ax.set_yticks(range(3)); ax.set_yticklabels([DISPLAY[c] for c in CORPORA])
    ax.set_xlabel("Evaluation corpus"); ax.set_ylabel("Training corpus")
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            value = mean.iloc[i, j]
            color = "white" if value < 0.55 else "black"
            ax.text(j, i, f"{value:.3f}\n$\\pm${std.iloc[i,j]:.3f}", ha="center", va="center",
                    fontsize=7.0, color=color)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Macro-F1")
    savefig(fig, stem)

heatmap_mean_std("fgm", "F18_cross_dataset_fgm_multiseed",
                 "FGM transfer across corpora (five seeds)")
heatmap_mean_std("vanilla", "F18_cross_dataset_vanilla_multiseed",
                 "Vanilla transfer across corpora (five seeds)")

# Canonical manuscript heatmap now means the primary FGM five-seed result, not the legacy single run.
d = summary[summary.system == "fgm"]
mean = d.pivot(index="source", columns="target", values="macro_f1_mean").loc[CORPORA, CORPORA]
std = d.pivot(index="source", columns="target", values="macro_f1_std").loc[CORPORA, CORPORA]
fig, ax = plt.subplots(figsize=(IN1, 3.05))
im = ax.imshow(mean.values, cmap="viridis", vmin=0.30, vmax=1.00, aspect="auto")
ax.set_xticks(range(3)); ax.set_xticklabels([DISPLAY[c] for c in CORPORA], rotation=25, ha="right")
ax.set_yticks(range(3)); ax.set_yticklabels([DISPLAY[c] for c in CORPORA])
ax.set_xlabel("Evaluation corpus"); ax.set_ylabel("Training corpus")
ax.set_title("Cross-corpus macro-F1: full fine-tuning + FGM")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{mean.iloc[i,j]:.3f}\n$\\pm${std.iloc[i,j]:.3f}",
                ha="center", va="center", fontsize=7.0,
                color="white" if mean.iloc[i,j] < 0.55 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Macro-F1")
savefig(fig, "F_cross_dataset_heatmap")


figure: F18_cross_dataset_fgm_multiseed
figure: F18_cross_dataset_vanilla_multiseed
figure: F_cross_dataset_heatmap


In [12]:
# Transfer gap, FGM effect, seed stability, and shift figures
# 1) In-domain vs transfer distributions across matched seeds.
plot_rows = []
for r in runs.itertuples():
    plot_rows.append(dict(system=r.system, regime="In-domain" if r.in_domain else "Cross-corpus",
                          macro_f1=r.test_macro_f1))
pdplot = pd.DataFrame(plot_rows)
fig, ax = plt.subplots(figsize=(IN1, 2.9))
positions = {("vanilla", "In-domain"):0.9, ("vanilla", "Cross-corpus"):1.2,
             ("fgm", "In-domain"):1.9, ("fgm", "Cross-corpus"):2.2}
rng = np.random.default_rng(42)
for (system, regime), g in pdplot.groupby(["system", "regime"]):
    x = positions[(system, regime)]
    color = C["blue"] if regime == "In-domain" else C["orange"]
    jitter = rng.normal(0, 0.025, len(g))
    ax.scatter(np.full(len(g), x)+jitter, g.macro_f1, s=13, alpha=0.65, color=color, edgecolor="none")
    ax.plot([x-0.09,x+0.09], [g.macro_f1.mean()]*2, color=C["dark"], lw=1.5)
ax.set_xticks([1.05,2.05]); ax.set_xticklabels(["Vanilla", "FGM"])
ax.set_ylabel("Macro-F1"); ax.set_ylim(0.25,1.02)
ax.set_title("In-domain accuracy does not imply cross-corpus robustness")
ax.legend(handles=[plt.Line2D([0],[0],marker='o',color='w',markerfacecolor=C['blue'],label='In-domain',markersize=5),
                   plt.Line2D([0],[0],marker='o',color='w',markerfacecolor=C['orange'],label='Cross-corpus',markersize=5)],
          loc="lower left", frameon=False)
savefig(fig, "F18_indomain_vs_transfer_multiseed", aliases=("F_indomain_vs_transfer",))

# 2) Forest plot of the matched seed effect of FGM.
order = paired.sort_values("delta_mean")
fig, ax = plt.subplots(figsize=(IN2, 3.4))
y = np.arange(len(order))
xerr = np.vstack([order.delta_mean-order.delta_ci95_lo, order.delta_ci95_hi-order.delta_mean])
colors = [C["green"] if v >= 0 else C["vermillion"] for v in order.delta_mean]
for i, (_, r) in enumerate(order.iterrows()):
    ax.errorbar(r.delta_mean, i, xerr=[[r.delta_mean-r.delta_ci95_lo],[r.delta_ci95_hi-r.delta_mean]],
                fmt='o', color=colors[i], ecolor=colors[i], capsize=2, markersize=4)
ax.axvline(0, color=C["gray"], lw=0.9, ls="--")
ax.set_yticks(y); ax.set_yticklabels([f"{DISPLAY[r.source]} $\\rightarrow$ {DISPLAY[r.target]}" for r in order.itertuples()])
ax.set_xlabel("FGM − vanilla macro-F1 (mean across matched seeds)")
ax.set_title("Effect of adversarial training across domains")
savefig(fig, "F18_fgm_vs_vanilla_transfer", aliases=("F_fgm_vs_vanilla_transfer",))

# 3) In-domain stability across corpora and systems.
ind = runs[runs.in_domain].copy()
fig, axes = plt.subplots(1, 3, figsize=(IN2, 2.55), sharey=True)
for ax, corpus in zip(axes, CORPORA):
    g = ind[ind.source == corpus]
    for system, marker, color in [("vanilla","o",C["blue"]),("fgm","s",C["green"])]:
        q = g[g.system == system].set_index("seed").loc[SEEDS]
        ax.plot(range(len(SEEDS)), q.test_macro_f1, marker=marker, color=color, lw=1.1, label=system.upper())
    ax.set_xticks(range(len(SEEDS))); ax.set_xticklabels([str(s) for s in SEEDS], rotation=30)
    ax.set_title(DISPLAY[corpus]); ax.set_xlabel("Seed")
axes[0].set_ylabel("In-domain macro-F1")
axes[-1].legend(frameon=False, loc="best")
fig.suptitle("Matched five-seed stability", y=1.02, fontsize=9.2, fontweight="bold")
savefig(fig, "F18_seed_stability", aliases=("F_seed_stability",))

# 4) Shift-vs-transfer is explicitly exploratory (six directed cells).
off = summary[(summary.system == "fgm") & (~summary.in_domain)].merge(
    shift, on=["source", "target", "in_domain"], how="left")
fig, ax = plt.subplots(figsize=(IN1, 2.8))
ax.scatter(off.char_tfidf_centroid_cosine, off.macro_f1_mean, s=35, color=C["purple"])
for r in off.itertuples():
    ax.annotate(f"{DISPLAY[r.source]}→{DISPLAY[r.target]}",
                (r.char_tfidf_centroid_cosine, r.macro_f1_mean), xytext=(3,3),
                textcoords="offset points", fontsize=6.2)
ax.set_xlabel("Character TF-IDF centroid cosine")
ax.set_ylabel("Transfer macro-F1 (five-seed mean)")
ax.set_title("Domain similarity and transfer performance (exploratory)")
savefig(fig, "F18_shift_vs_transfer", aliases=("F_shift_vs_transfer",))

# 5) Validation-fitted temperature scaling, separated by evaluation regime.
cal_rows=[]
for (system, regime), g in runs.assign(
        regime=np.where(runs.in_domain, "In-domain", "Cross-corpus")
    ).groupby(["system", "regime"]):
    cal_rows.append(dict(system=system, regime=regime,
                         raw=float(g.ece.mean()), scaled=float(g.ece_temperature_scaled.mean())))
cal=pd.DataFrame(cal_rows)
fig,axes=plt.subplots(1,2,figsize=(IN2,2.65),sharey=True)
for ax,system in zip(axes,["vanilla","fgm"]):
    q=cal[cal.system==system].set_index("regime").loc[["In-domain","Cross-corpus"]]
    x=np.arange(2); w=.34
    ax.bar(x-w/2,q.raw,w,label="Uncalibrated",color=C["orange"])
    ax.bar(x+w/2,q.scaled,w,label="Temperature-scaled",color=C["green"])
    ax.set_xticks(x); ax.set_xticklabels(q.index); ax.set_title(system.upper())
    ax.set_ylabel("Expected calibration error")
handles,labels=axes[-1].get_legend_handles_labels()
fig.legend(handles,labels,loc="lower center",bbox_to_anchor=(0.5,-0.08),ncol=2,frameon=False)
fig.subplots_adjust(bottom=0.24)
fig.suptitle("Calibration under domain shift (validation-fitted temperature)",
             y=1.02,fontsize=9.2,fontweight="bold")
savefig(fig,"F18_cross_domain_calibration")

# 6) Class-specific recall under transfer for the primary FGM system.
q=summary[(summary.system=="fgm") & (~summary.in_domain)].groupby("target")[
    ["recall_class_0_mean","recall_class_1_mean"]].mean().reindex(CORPORA)
fig,ax=plt.subplots(figsize=(IN1,2.65)); x=np.arange(3); w=.36
ax.bar(x-w/2,q.recall_class_0_mean,w,label="Non-sarcastic",color=C["blue"])
ax.bar(x+w/2,q.recall_class_1_mean,w,label="Sarcastic",color=C["orange"])
ax.set_xticks(x); ax.set_xticklabels([DISPLAY[c] for c in CORPORA],rotation=15)
ax.set_ylim(0,1); ax.set_ylabel("Recall")
ax.set_title("Class-specific recall under cross-corpus transfer")
ax.legend(frameon=False)
savefig(fig,"F18_transfer_recall_asymmetry",aliases=("F_error_asymmetry",))


figure: F18_indomain_vs_transfer_multiseed
figure: F18_fgm_vs_vanilla_transfer
figure: F18_seed_stability
figure: F18_shift_vs_transfer
figure: F18_cross_domain_calibration
figure: F18_transfer_recall_asymmetry


In [ ]:
# Regenerate supporting figures only from verified canonical inputs.
def table_path(name):
    p = TABLES / name
    return p if p.exists() else None

# Class distribution from the actual split files.
fig, ax = plt.subplots(figsize=(IN1, 2.65))
class_rows=[]
for c in CORPORA:
    full=pd.concat([DATA[c][s] for s in ("train","val","test")],ignore_index=True)
    vc=full.label_binary.value_counts().reindex([0,1],fill_value=0)
    class_rows.append((c,vc[0],vc[1]))
x=np.arange(3); w=.36
ax.bar(x-w/2,[r[1] for r in class_rows],w,label="Non-sarcastic",color=C["blue"])
ax.bar(x+w/2,[r[2] for r in class_rows],w,label="Sarcastic",color=C["orange"])
ax.set_xticks(x); ax.set_xticklabels([DISPLAY[r[0]] for r in class_rows],rotation=15)
ax.set_ylabel("Documents"); ax.set_title("Class distribution after cleaning")
ax.legend(frameon=False)
savefig(fig,"F_class_distribution")

# Reproduction fidelity: reproduced vs reported on the same Ben-Sarc split.
p=table_path("03_dl_reference_gap_same_split.csv")
if p:
    d=pd.read_csv(p).sort_values("test_macro_f1")
    fig,ax=plt.subplots(figsize=(IN2,3.0)); y=np.arange(len(d)); h=.34
    ax.barh(y-h/2,d.reference_f1,h,label="Reported",color=C["gray"])
    ax.barh(y+h/2,d.test_macro_f1,h,label="Reproduced",color=C["blue"])
    ax.set_yticks(y); ax.set_yticklabels(d.label); ax.set_xlabel("Macro-F1")
    ax.set_title("Deep-model reproduction fidelity on the common split")
    ax.legend(loc="lower center",bbox_to_anchor=(0.5,-0.26),ncol=2,frameon=False)
    fig.subplots_adjust(bottom=0.25)
    savefig(fig,"F_repro_fidelity")

# Multi-backbone heatmap; each cell is explicitly a single run from Notebook 06.
p=table_path("06_multi_backbone_summary.csv")
if p:
    d=pd.read_csv(p); d=d[d.task.isin(CORPORA)]
    piv=d.pivot(index="backbone",columns="task",values="test_macro_f1")
    piv=piv.reindex(columns=CORPORA)
    fig,ax=plt.subplots(figsize=(IN2,3.1)); im=ax.imshow(piv.values,cmap="YlGnBu",vmin=.60,vmax=1.0,aspect="auto")
    ax.set_xticks(range(3)); ax.set_xticklabels([DISPLAY[c] for c in CORPORA])
    ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            if np.isfinite(piv.iloc[i,j]): ax.text(j,i,f"{piv.iloc[i,j]:.3f}",ha="center",va="center",fontsize=6.8)
    ax.set_title("Backbone comparison (single seed per cell)"); fig.colorbar(im,ax=ax,label="Macro-F1")
    savefig(fig,"F_multibackbone")

# Adversarial methods; single-seed label is deliberate.
p=table_path("07_adversarial_summary.csv")
if p:
    d=pd.read_csv(p); d=d[d.task.isin(CORPORA)]
    piv=d.pivot(index="method",columns="task",values="test_macro_f1").reindex(columns=CORPORA)
    fig,ax=plt.subplots(figsize=(IN2,2.9)); im=ax.imshow(piv.values,cmap="YlOrBr",vmin=.60,vmax=1.0,aspect="auto")
    ax.set_xticks(range(3)); ax.set_xticklabels([DISPLAY[c] for c in CORPORA])
    ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]): ax.text(j,i,f"{piv.iloc[i,j]:.3f}",ha="center",va="center",fontsize=7)
    ax.set_title("Adversarial-training ablation (single seed)"); fig.colorbar(im,ax=ax,label="Macro-F1")
    savefig(fig,"F_adversarial")

# Existing enhanced-head calibration remains exploratory and single-run.
p=table_path("13_calibration.csv")
if p:
    d=pd.read_csv(p); x=np.arange(len(d)); w=.35
    fig,ax=plt.subplots(figsize=(IN1,2.65))
    ax.bar(x-w/2,d.ece,w,label="Before scaling",color=C["orange"])
    ax.bar(x+w/2,d.ece_after,w,label="After scaling",color=C["green"])
    labels=["Enhanced FGM+AWP\n(single run)" if "09b" in str(m) else "Vanilla baseline\n(single run)" for m in d.model]
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Expected calibration error")
    ax.set_title("Post-hoc temperature scaling (exploratory)"); ax.legend(frameon=False)
    savefig(fig,"F_calibration")

p=FT/"18_llm_baselines.csv"
if not p.exists(): p=TABLES/"18_llm_baselines.csv"
if p.exists():
    d=pd.read_csv(p); d=d[d.task.isin(CORPORA)]
    fig,ax=plt.subplots(figsize=(IN2,3.0)); tasks=CORPORA; x=np.arange(len(tasks)); w=.34
    for i,(setting,color) in enumerate([("zero_shot",C["gray"]),("few_shot",C["purple"])]):
        q=d[d.setting==setting].set_index("task").reindex(tasks)
        y=q.macro_f1.values; lo=y-q.ci95_lo.values; hi=q.ci95_hi.values-y
        ax.bar(x+(i-.5)*w,y,w,label=setting.replace("_"," ").title(),color=color,
               yerr=np.vstack([lo,hi]),capsize=2)
    ax.set_xticks(x); ax.set_xticklabels([DISPLAY[c] for c in tasks]); ax.set_ylabel("Macro-F1")
    ax.set_ylim(0,1); ax.set_title("Prompt-based LLM baselines"); ax.legend(frameon=False)
    savefig(fig,"F_llm_baselines")


figure: F_class_distribution
figure: F_repro_fidelity
figure: F_multibackbone
figure: F_adversarial
figure: F_calibration
figure: F_llm_baselines


In [ ]:
# Publish canonical tables to finalized_outputs and write the final manifest.
canonical_tables = [
    RUN_TABLE, SUMMARY_TABLE, TEST_TABLE, GAP_TABLE, SHIFT_TABLE,
    TABLES / "18_split_audit.csv", TABLES / "18_shift_transfer_association.json",
    TABLES / "18_artifact_inventory.csv", TABLES / "18_artifact_decisions.csv",
]
for src in canonical_tables:
    if src.exists():
        shutil.copy2(src, FT / src.name)

config = dict(
    notebook="18_reviewer_ready_multiseed_cross_corpus.ipynb",
    created_utc=datetime.now(timezone.utc).isoformat(), root=str(ROOT), model_name=MODEL_NAME,
    architecture="BanglaBERT standard sequence-classification head",
    systems=SYSTEMS, corpora=CORPORA, seeds=SEEDS, max_length=MAX_LENGTH,
    epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE, eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
    leakage_policy="off-diagonal target test excludes normalized exact overlaps with source train+val+test",
    primary_uncertainty="Student-t 95% CI across five training seeds",
    paired_test="exact sign-flip permutation over matched seed differences; Holm over 9 cells",
    gpu=GPU_NAME, cuda=torch.version.cuda, torch=torch.__version__, transformers=transformers.__version__,
    pandas=pd.__version__, scipy=scipy.__version__,
    elapsed_wall_seconds=float(time.time()-NOTEBOOK_START),
    exclusion_policy={
        "09b_fgm_awp_multiseed.csv":"one enhanced-head run; not multiseed",
        "17_proposed_headline.csv":"architecture conflation; excluded",
        "10_cross_dataset_matrix.csv":"legacy single-seed transfer; superseded",
    },
)
json.dump(config, open(TABLES / "18_run_config.json", "w"), indent=2)
shutil.copy2(TABLES / "18_run_config.json", FT / "18_run_config.json")

# Refresh the inventory once so newly generated canonical JSON/CSV files are represented in the manifest.
manifest = dict(
    notebook=config["notebook"], status="complete",
    expected_training_jobs=len(SYSTEMS)*len(CORPORA)*len(SEEDS),
    completed_training_jobs=int(runs.groupby(["system","source","seed"]).ngroups),
    expected_source_target_rows=expected_rows, observed_source_target_rows=len(runs),
    canonical_tables=sorted(p.name for p in FT.glob("18_*.csv")) + sorted(p.name for p in FT.glob("18_*.json")),
    generated_figures=sorted(set(generated_figures)),
    figure_files=sorted(p.name for p in FF.glob("F*.pdf") if p.stem in set(generated_figures)),
    checkpoint_policy="one best-validation checkpoint per system/source; temporary trainer states removed after success",
    primary_system="plain full fine-tuning + FGM; enhanced FGM+AWP head remains exploratory",
    all_csv_json_audited=True,
)
json.dump(manifest, open(FINAL / "MANIFEST_sha.json", "w"), indent=2)
print(json.dumps(manifest, indent=2))


{
  "notebook": "18_reviewer_ready_multiseed_cross_corpus.ipynb",
  "status": "complete",
  "expected_training_jobs": 30,
  "completed_training_jobs": 30,
  "expected_source_target_rows": 90,
  "observed_source_target_rows": 90,
  "canonical_tables": [
    "18_artifact_decisions.csv",
    "18_artifact_inventory.csv",
    "18_cross_corpus_shift_audit.csv",
    "18_fgm_vs_vanilla_paired_seed_tests.csv",
    "18_llm_baselines.csv",
    "18_split_audit.csv",
    "18_transfer_gaps_by_seed.csv",
    "18_transformer_cross_corpus_multiseed_runs.csv",
    "18_transformer_cross_corpus_multiseed_summary.csv",
    "18_run_config.json",
    "18_shift_transfer_association.json"
  ],
  "generated_figures": [
    "F18_cross_dataset_fgm_multiseed",
    "F18_cross_dataset_vanilla_multiseed",
    "F18_cross_domain_calibration",
    "F18_fgm_vs_vanilla_transfer",
    "F18_indomain_vs_transfer_multiseed",
    "F18_seed_stability",
    "F18_shift_vs_transfer",
    "F18_transfer_recall_asymmetry",
    "F_adv

## Interpretation guardrails for the manuscript

After the run, manuscript numbers should come from
`18_transformer_cross_corpus_multiseed_summary.csv`. Report the mean, standard deviation, and 95% CI
across seeds. Treat `18_fgm_vs_vanilla_paired_seed_tests.csv` as the matched comparison and state that
Holm correction covers the nine directed source-target cells.

Do **not** relabel the five plain-FGM seeds as FGM+AWP. The enhanced 0.8038 result remains one
exploratory run. Do **not** present the shift correlation as causal evidence; it contains only six
directed off-diagonal points.
